# Recipe RecSys | 5,000 sampled users, Top2000 recipes

Compare Popular, implicit Item-KNN and LightFM with ingredient features.
Metrics are macro Precision@N, Recall@N and F1@N.


In [46]:
import os
import time
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from lightfm import LightFM
import warnings
warnings.filterwarnings('ignore')

SEED = 42
SAMPLE_USERS = 5000
TEST_FRACTION = 0.1
NS = [3, 5, 10]


NUM_THREADS = os.cpu_count()

## Load and sample data


In [47]:
top2000 = pd.read_csv('RecipesData/top2000/top2000_recipe_data.txt', sep='\t', 
                      header=None, names=['recipe', 'ingredients', 'procedure'])

raw = pd.read_csv('RecipesData/user_recipe_box_tab_main_meat.csv', sep='\t', 
                  header=None, usecols=[0, 1], names=['user', 'recipe'])

raw = raw[raw['recipe'].isin(top2000['recipe'])].drop_duplicates(['user', 'recipe'])
rng = np.random.default_rng(SEED)
users = rng.choice(raw['user'].unique(), min(SAMPLE_USERS, raw['user'].nunique()), replace=False)
raw = raw[raw['user'].isin(users)].copy()


In [48]:
raw.head()

,user,recipe
3820,10002095,addictive-sweet-potato-burritos
3821,10002095,aimees-quick-chicken
3822,10002095,alfredo-mostaccioli
3823,10002095,all-american-roast-beef
3824,10002095,ambers-sesame-chicken


In [49]:
users = pd.Index(sorted(raw['user'].unique()))
item_names = pd.Index(top2000['recipe'].drop_duplicates())

# create indicies for users and items
user_ids = {name: idx for idx, name in enumerate(users)}
item_ids = {name: idx for idx, name in enumerate(item_names)}
raw['u'] = raw['user'].map(user_ids)
raw['i'] = raw['recipe'].map(item_ids)

n_users, n_items = len(users), len(item_names)
print(f'{n_users:,} users, {n_items:,} recipes, {len(raw):,} positive interactions')

5,000 users, 2,000 recipes, 155,188 positive interactions


For each user with at least two likes, hold out approximately 10% to use in test. Users with only one like remain in training and are excluded from evaluation.
This measures recommendations for existing users, not cold-start or temporal performance.


In [50]:
rng = np.random.default_rng(SEED)

test_idx = []

for u, group in raw.groupby('u'):
    n = len(group)

    if n > 1:
        k = int(np.ceil(n * TEST_FRACTION))
        test_idx.extend(rng.choice(group.index, size=k, replace=False))

train_df = raw.drop(test_idx)
test_df = raw.loc[test_idx]

train = sparse.csr_matrix(
    (np.ones(len(train_df)), (train_df['u'], train_df['i'])),
    shape=(n_users, n_items)
)

test = sparse.csr_matrix(
    (np.ones(len(test_df)), (test_df['u'], test_df['i'])),
    shape=(n_users, n_items)
)

In [51]:
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (5000, 2000)
Test shape: (5000, 2000)


## Models

### A. Popular baseline

Popularity is computed from training positives.


In [52]:
item_pop = np.asarray(train.sum(axis=0)).ravel()
pop_ratings = np.broadcast_to(item_pop, (n_users, n_items))


In [53]:
pop_ratings

array([[818., 644., 529., ...,  26.,  23.,  30.],
       [818., 644., 529., ...,  26.,  23.,  30.],
       [818., 644., 529., ...,  26.,  23.,  30.],
       ...,
       [818., 644., 529., ...,  26.,  23.,  30.],
       [818., 644., 529., ...,  26.,  23.,  30.],
       [818., 644., 529., ...,  26.,  23.,  30.]])

In [54]:
top_indices = np.argsort(-item_pop)[:10]

print("10 most popular recipes\n")

for idx in top_indices:
    recipe_name = item_names[idx]
    likes = item_pop[idx]
    print(f"{recipe_name}: {likes}")

10 most popular recipes

worlds-best-lasagna: 818.0
chicken-pot-pie-ix: 644.0
awesome-slow-cooker-pot-roast: 529.0
chicken-cordon-bleu-ii: 495.0
broiled-tilapia-parmesan: 460.0
baked-teriyaki-chicken: 424.0
chicken-marsala: 399.0
zesty-slow-cooker-chicken-barbecue: 391.0
roast-sticky-chicken-rotisserie-style: 391.0
bbq-pork-for-sandwiches: 386.0


### B. Implicit Item-KNN (k=5)

Compute cosine similarity using training positives. For each target recipe,
keep its five most similar recipes.


In [55]:
def item_knn_predict(train, k=5):

    similarity = cosine_similarity(train.T).astype(np.float32)
    np.fill_diagonal(similarity, 0)
    
    neighbors = np.argsort(-similarity, axis=1, kind='stable')[:, :k]

    mask = np.zeros(similarity.shape)
    for i in range(similarity.shape[0]):
        mask[i, neighbors[i]] = similarity[i, neighbors[i]]

    return (train @ mask.T)

knn_ratings = item_knn_predict(train, k=5)


### C. LightFM with ingredient features


In [56]:
recipes = pd.read_csv('RecipesData/recipe_ingr.csv')
recipes = recipes.set_index('recipe_name').loc[item_names]
print(recipes.head())


                               adobo_sauce  adobo_seasoning  ahi_tuna_steak  \
recipe                                                                        
worlds-best-lasagna                      0                0               0   
chicken-pot-pie-ix                       0                0               0   
awesome-slow-cooker-pot-roast            0                0               0   
chicken-cordon-bleu-ii                   0                0               0   
broiled-tilapia-parmesan                 0                0               0   

                               alfalfa_sprout  alfredo_sauce  \
recipe                                                         
worlds-best-lasagna                         0              0   
chicken-pot-pie-ix                          0              0   
awesome-slow-cooker-pot-roast               0              0   
chicken-cordon-bleu-ii                      0              0   
broiled-tilapia-parmesan                    0              0  

In [57]:
# Each recipe is represented by it's identity and ingredients
item_features = sparse.hstack([
    sparse.eye(n_items),
    sparse.csr_matrix(recipes.values)
]).tocsr()

item_features = normalize(item_features, norm='l2', axis=1)


In [58]:
lfm = LightFM(
    no_components=64,
    loss='warp',
    random_state=SEED,
    learning_rate=0.05,)

lfm.fit(train, item_features=item_features, epochs = 30, verbose = True)

lfm_ratings = np.array([
    lfm.predict(u, np.arange(n_items), item_features = item_features)
    for u in range(n_users)])

Epoch: 100%|██████████| 30/30 [00:10<00:00,  2.91it/s]


## Evaluation on the full unseen catalog

All models are compared using precision@N, recall@N and F1-score@N metrics. 

- "relevant" recipes are the ones person liked and come from test
- "recommended" recipes are top-N predictions from the model
- "hits" is the intersection of the two above


In [59]:
def evaluate(ratings, train, test, top_n):
    precisions, recalls, f1s = [], [], []

    for u in range(train.shape[0]):
        relevant = test[u].indices

        if len(relevant) == 0:
            continue

        # Exclude recipes seen during training
        scores = ratings[u].copy()
        scores[train[u].indices] = -np.inf

        # Top-N recommendations
        recommended = np.argsort(-scores)[:top_n]

        hits = len(set(recommended) & set(relevant))

        precision = hits / top_n
        recall = hits / len(relevant)
        f1 = 2 * precision * recall / (precision + recall) if hits else 0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return {
        'Precision': np.mean(precisions),
        'Recall': np.mean(recalls),
        'F1': np.mean(f1s)
        }

results = []

for name, scores in [
    ('Popular', pop_ratings),
    ('Item-KNN (k=5)', knn_ratings),
    ('LightFM-Hybrid', lfm_ratings)]:
    
    for n in NS:
        metrics = evaluate(scores, train, test, n)
        results.append({'Model': name, 'N': n, **metrics})

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))


         Model  N  Precision  Recall     F1
       Popular  3     0.0289  0.0311 0.0237
       Popular  5     0.0248  0.0413 0.0243
       Popular 10     0.0197  0.0608 0.0238
Item-KNN (k=5)  3     0.0257  0.0227 0.0188
Item-KNN (k=5)  5     0.0223  0.0321 0.0206
Item-KNN (k=5) 10     0.0174  0.0483 0.0204
LightFM-Hybrid  3     0.0306  0.0276 0.0227
LightFM-Hybrid  5     0.0263  0.0368 0.0238
LightFM-Hybrid 10     0.0224  0.0606 0.0258


## Interpretation

Recipes data could be subject to "popularity bias", meaning there are recipes that are favoured by many (and are probably easy to make). For example, chicken, cupcakes, sandwiches, which everyone likes regardless of personal tastes. This could explain why baseline Popular recommender has relatively high metrics.

Item-based kNN could suffer from the "tunnel vision" problem. Since the model relies heavily on items' ingredients, it tends to recommend dishes that match previous actions. If a person has ingredients in the fridge to make a dish they already cooked, they will probably stick to it, instead of making something new. 

LightFM shows similar results to Popular recommender and with hyperparameter tuning could be better.

In [60]:
# decrease lr and increase epochs

lfm = LightFM(
    no_components=64,
    loss='warp',
    random_state=SEED,
    learning_rate=0.01,)

lfm.fit(train, item_features=item_features, epochs = 60, verbose = True)

lfm_ratings = np.array([
    lfm.predict(u, np.arange(n_items), item_features = item_features)
    for u in range(n_users)])

Epoch: 100%|██████████| 60/60 [00:21<00:00,  2.85it/s]


In [61]:
for n in NS:
    metrics = evaluate(lfm_ratings, train, test, n)

    print(f"N = {n}, {metrics}")

# The model is slightly better for lower Ns

N = 3, {'Precision': 0.0302496328928047, 'Recall': 0.028361269288514467, 'F1': 0.02292901668327329}
N = 5, {'Precision': 0.02625550660792952, 'Recall': 0.03929094057473895, 'F1': 0.024482304545785313}
N = 10, {'Precision': 0.02182819383259912, 'Recall': 0.06098700012165034, 'F1': 0.02540361672724234}


In [62]:
# decrease number of components, since the dataset may be not sufficiently large

lfm = LightFM(
    no_components=32,
    loss='warp',
    random_state=SEED,
    learning_rate=0.01,)

lfm.fit(train, item_features=item_features, epochs = 60, verbose = True)

lfm_ratings = np.array([
    lfm.predict(u, np.arange(n_items), item_features = item_features)
    for u in range(n_users)])

Epoch: 100%|██████████| 60/60 [00:11<00:00,  5.17it/s]


In [63]:
for n in NS:
    metrics = evaluate(lfm_ratings, train, test, n)

    print(f"N = {n}, {metrics}")

N = 3, {'Precision': 0.028854625550660793, 'Recall': 0.026848537448739544, 'F1': 0.021873950363974663}
N = 5, {'Precision': 0.02524229074889868, 'Recall': 0.03584728142097276, 'F1': 0.02325612834776799}
N = 10, {'Precision': 0.020947136563876655, 'Recall': 0.055939029448159545, 'F1': 0.024211845511081985}


In [64]:
lfm = LightFM(
    no_components=64,
    loss='warp',
    random_state=SEED,
    learning_rate=0.01,)

lfm.fit(train, item_features=item_features, epochs = 100, verbose = True)

lfm_ratings = np.array([
    lfm.predict(u, np.arange(n_items), item_features = item_features)
    for u in range(n_users)])

Epoch: 100%|██████████| 100/100 [00:34<00:00,  2.90it/s]


In [65]:
for n in NS:
    metrics = evaluate(lfm_ratings, train, test, n)

    print(f"N = {n}, {metrics}")

# With low learning rate and sufficiently large epochs the model converges to a better optima

N = 3, {'Precision': 0.03201174743024963, 'Recall': 0.029575360468932093, 'F1': 0.024028707179204672}
N = 5, {'Precision': 0.02731277533039648, 'Recall': 0.04024224125795489, 'F1': 0.025208825263555446}
N = 10, {'Precision': 0.02198237885462555, 'Recall': 0.061352632874942234, 'F1': 0.02554187211358629}
